# Étape 4 — Validation

On vérifie chaque candidat : est-ce que la confidence >= (1 - epsilon) ?
C'est ici qu'on sépare les vraies PFDs du bruit.

In [2]:
import sys
sys.path.insert(0, '..')
import pandas as pd
from pfd_verifier import verifier_pfd, rapport_pfd

In [3]:
t2 = pd.read_csv('../data/pfd_validation/t2.csv')

## Tester des PFDs manuellement

In [4]:
# : prefix(ZIP, 3) = '606' -> CITY = 'Chicago'
rapport_pfd(t2, col_X='ZIP', pattern_X='606', col_Y='CITY', pattern_Y='Chicago',
            epsilon=0.1, match_type_X='startswith', match_type_Y='exact')

  ZIP [startswith: '606'] -> CITY [exact: 'Chicago']
  epsilon=0.1  |  seuil=90.0%
------------------------------------------------------------
  Support (X matche)  : 2,131
  Valides (X et Y)    : 2,094
  Violations          : 37
  Confiance           : 98.26%
  Résultat            : VALIDE
------------------------------------------------------------
  Exemples de violations :
    [1] ZIP='60622'  |  CITY='chicago'
    [2] ZIP='60602'  |  CITY='chicago'
    [3] ZIP='60602'  |  CITY='chicago'



{'confidence': 0.982637,
 'is_valid': True,
 'n_matching_X': 2131,
 'n_valid': 2094,
 'n_violations': 37,
 'epsilon': 0.1,
 'examples_violations': [{'Year': 2015.0,
   'EMPLOYER_ID': 6801,
   'NAME': 'george K. baum & company',
   'ADDRESS_1': '1 N. La Salle St',
   'ADDRESS_2': 'suite 1725',
   'CITY': 'chicago',
   'STATE': 'IL',
   'ZIP': '60622',
   'COUNTRY': 'United States',
   'PHONE': '312-641-3611',
   'FAX': '312-641-3625',
   'CREATED_DATE': '12/27/2012',
   'ACTIVE': 'Y'},
  {'Year': 2015.0,
   'EMPLOYER_ID': 6881,
   'NAME': 'george K. baum & company',
   'ADDRESS_1': '1 N. La Salle St',
   'ADDRESS_2': 'suite 1725',
   'CITY': 'chicago',
   'STATE': 'IL',
   'ZIP': '60602',
   'COUNTRY': 'United States',
   'PHONE': '312-641-3611',
   'FAX': '312-641-3625',
   'CREATED_DATE': '01/02/2013',
   'ACTIVE': 'Y'},
  {'Year': 2014.0,
   'EMPLOYER_ID': 6881,
   'NAME': 'george K. baum & company',
   'ADDRESS_1': '1 N. La Salle St',
   'ADDRESS_2': 'suite 1725',
   'CITY': 'chicag

In [5]:
#  prefix(ZIP, 3) = '606' -> STATE = 'CA'
rapport_pfd(t2, col_X='ZIP', pattern_X='606', col_Y='STATE', pattern_Y='CA',
            epsilon=0.1, match_type_X='startswith', match_type_Y='exact')

  ZIP [startswith: '606'] -> STATE [exact: 'CA']
  epsilon=0.1  |  seuil=90.0%
------------------------------------------------------------
  Support (X matche)  : 2,131
  Valides (X et Y)    : 0
  Violations          : 2,131
  Confiance           : 0.00%
  Résultat            : INVALIDE
------------------------------------------------------------
  Exemples de violations :
    [1] ZIP='60603'  |  STATE='Il'
    [2] ZIP='60602'  |  STATE='IL'
    [3] ZIP='60602'  |  STATE='IL'



{'confidence': 0.0,
 'is_valid': False,
 'n_matching_X': 2131,
 'n_valid': 0,
 'n_violations': 2131,
 'epsilon': 0.1,
 'examples_violations': [{'Year': 2011.0,
   'EMPLOYER_ID': 4818,
   'NAME': 'Lighten-Gale LLC',
   'ADDRESS_1': '39 S. LaSalle St., Ste. 808',
   'ADDRESS_2': nan,
   'CITY': 'Chicago',
   'STATE': 'Il',
   'ZIP': '60603',
   'COUNTRY': 'United States',
   'PHONE': '312-920-1500',
   'FAX': nan,
   'CREATED_DATE': nan,
   'ACTIVE': 'Y'},
  {'Year': 2012.0,
   'EMPLOYER_ID': 4099,
   'NAME': 'Nicolay & Dart LLC',
   'ADDRESS_1': '33 N. Dearborn St. Ste.2200',
   'ADDRESS_2': nan,
   'CITY': 'Chicago',
   'STATE': 'IL',
   'ZIP': '60602',
   'COUNTRY': 'United States',
   'PHONE': '312-701-0221',
   'FAX': '312-658-0464',
   'CREATED_DATE': nan,
   'ACTIVE': 'Y'},
  {'Year': 2011.0,
   'EMPLOYER_ID': 4099,
   'NAME': 'Nicolay & Dart LLC',
   'ADDRESS_1': '33 N. Dearborn St. Ste.2200',
   'ADDRESS_2': nan,
   'CITY': 'Chicago',
   'STATE': 'IL',
   'ZIP': '60602',
   'COU

## Valider les candidats de l'étape 3

In [6]:
from src.pattern_extractor import extract_all_patterns
from src.candidate_generator import generate_all_candidates

pattern_maps = {}
for col in t2.columns:
    pmap = extract_all_patterns(t2[col], k_max=3, min_support=30)
    if pmap:
        pattern_maps[col] = pmap

candidates = generate_all_candidates(t2, pattern_maps, min_support=30)

# valider chacun
valid = []
for cand in candidates:
    res = verifier_pfd(t2,
        col_X=cand['col_X'], pattern_X=cand['pattern_X'],
        col_Y=cand['col_Y'], pattern_Y=cand['pattern_Y'],
        epsilon=0.1,
        match_type_X=cand['match_type_X'], match_type_Y=cand['match_type_Y'])
    if res['is_valid']:
        cand.update(res)
        valid.append(cand)

print(f'{len(valid)} PFDs valides sur {len(candidates)} candidats')
for v in valid[:15]:
    print(f"  {v['col_X']}[{v['match_type_X']}:'{v['pattern_X']}'] -> {v['col_Y']}={v['pattern_Y']!r}  conf={v['confidence']:.1%}  support={v['n_matching_X']}")

1087 PFDs valides sur 5061 candidats
  Year[startswith:'2'] -> COUNTRY='United States'  conf=99.6%  support=3321
  Year[startswith:'2'] -> ACTIVE='Y'  conf=96.9%  support=3321
  Year[startswith:'20'] -> COUNTRY='United States'  conf=99.6%  support=3321
  Year[startswith:'20'] -> ACTIVE='Y'  conf=96.9%  support=3321
  Year[startswith:'201'] -> COUNTRY='United States'  conf=99.6%  support=3321
  Year[startswith:'201'] -> ACTIVE='Y'  conf=96.9%  support=3321
  Year[startswith:'2011'] -> COUNTRY='United States'  conf=100.0%  support=711
  Year[startswith:'2011'] -> ACTIVE='Y'  conf=92.7%  support=711
  Year[startswith:'2012'] -> COUNTRY='United States'  conf=99.9%  support=768
  Year[startswith:'2012'] -> ACTIVE='Y'  conf=93.4%  support=768
  Year[startswith:'2013'] -> COUNTRY='United States'  conf=99.2%  support=512
  Year[startswith:'2013'] -> ACTIVE='Y'  conf=100.0%  support=512
  Year[startswith:'2015'] -> COUNTRY='United States'  conf=99.8%  support=428
  Year[startswith:'2015'] -> AC